In [1]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
from sklearn.manifold import TSNE
import glasbey
import os

pio.renderers.default = 'browser'

In [ ]:
# CHANGE HERE: Path to your CSV file - Comment out
csv_file = r'C:\Users\gangliagurdian\Desktop\Mias Folder\new graphs - AC 9152025\data\combined_matrix_final_total_mp.csv'

combined_matrix = pd.read_csv(csv_file)

# Choose output path
save_path = r"D:\AccelClusterVisualizations\data"

In [3]:
# Compute t-SNE embedding (if not precomputed)
features = combined_matrix.iloc[:, 0:30].values.astype(float)
tsne = TSNE(n_components=2, perplexity=30, learning_rate=100, random_state=42)
Y = tsne.fit_transform(features)
combined_matrix['TSNE-1'] = Y[:, 0]
combined_matrix['TSNE-2'] = Y[:, 1]

# Get all clusters present across the ENTIRE dataset
all_clusters = sorted(combined_matrix['Cluster'].unique())
cluster_labels_str = [str(c) for c in all_clusters]

In [4]:
# CHANGE HERE: Customize palette size to the number of clusters - Make it automatically count number of clusters
glasbey_palette = glasbey.create_palette(palette_size=97)
color_num = len(glasbey_palette)
cluster_to_color = {
    str(c): glasbey_palette[i % color_num]
    for i, c in enumerate(all_clusters)
}

In [ ]:
# 1. Master t-SNE plot (all data, colored consistently)
fig_all = px.scatter(
    combined_matrix,
    x='TSNE-1', y='TSNE-2',
    color=combined_matrix['Cluster'].astype(str),
    color_discrete_map=cluster_to_color,
    hover_name=combined_matrix['Cluster'].astype(str),
    labels={'color': 'Cluster'},
    title='t-SNE of All Data'
)
fig_all.update_traces(marker=dict(size=7, line=dict(width=0.5, color='DarkSlateGrey')))
fig_all.update_layout(
    legend_title='Cluster',
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False)
)

x_range = fig_all.layout.xaxis.range # Capture axis ranges from master plot
y_range = fig_all.layout.yaxis.range

save_path_all = os.path.join(save_path, f"tSNE_all")
pio.write_image(fig_all, save_path_all + ".png", width=1000, height=1000)
pio.write_image(fig_all, save_path_all + ".svg", width=1000, height=1000)

fig_all.show() 

In [ ]:
# 2-6. Per-experiment type t-SNE plots, always using the full color map
# CHANGE HERE: Define exp_label to update titles of your plots

# Make stage names automated
for exp_type, exp_label in zip([8, 10, 12, 14, 23], ['Stage 1', 'Stage 2', 'Stage 3', "Stage 4", "Stage 5"]):
    combined_matrix_exp = combined_matrix[combined_matrix['Week_Number'] == exp_type].copy()
    if combined_matrix_exp.empty:
        continue
    fig = px.scatter(
        combined_matrix_exp,
        x='TSNE-1', y='TSNE-2',
        color=combined_matrix_exp['Cluster'].astype(str),
        color_discrete_map=cluster_to_color,
        category_orders={'Cluster': cluster_labels_str},
        hover_name=combined_matrix_exp['Cluster'].astype(str),
        labels={'color': 'Cluster'},
        title=f't-SNE: {exp_label}'
    )
    fig.update_traces(marker=dict(size=7, line=dict(width=0.5, color='DarkSlateGrey')))
    fig.update_layout(
        legend_title='Cluster',
        plot_bgcolor='white',
        paper_bgcolor='white',
        xaxis=dict(showgrid=False, range=x_range),
        yaxis=dict(showgrid=False, range=y_range)
    )

    save_path_individual = os.path.join(save_path, f"tSNE_{exp_label}")
    pio.write_image(fig, save_path_individual + ".png", width=1000, height=1000)
    pio.write_image(fig, save_path_individual + ".svg", width=1000, height=1000)


    fig.show()